In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec
import seaborn as sns
import pandas as pd
import random
import hydra
import lightning as lit
import torch
import torch.nn as nn
import wandb
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from lightning.pytorch.callbacks import LearningRateMonitor, EarlyStopping, RichProgressBar, ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger, CSVLogger
from loguru import logger
from omegaconf import DictConfig, OmegaConf
from torch.utils.data import DataLoader
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from omegaconf import OmegaConf

from datetime import datetime

from machine_intelligence.models.SAnD import SAnD
from machine_intelligence.utilities.data_handling import load_parquet_table
from machine_intelligence.data.SepsisPredDataModule import SepsisPredDataModule
from machine_intelligence.data.TransformsData import ZeroFill
# from machine_intelligence.interpretability.shap_analysis import ModelShapWrapper

from machine_intelligence.models.GatedTransformer import GatedTransformer

from machine_intelligence.utilities.viz_setup import figure_setup

import shap


In [ ]:
cwd = Path.cwd()
cwd


In [ ]:
project_root = Path(cwd).parents[1]
project_root


In [ ]:
current_date = datetime.now().strftime("%Y-%m-%d")
plot_output_dir = project_root / "plot_outputs" / current_date
plot_output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
model_path = project_root / Path("experiment_logs/sand-debug/version_197/checkpoints/epoch=19-step=620.ckpt")
model = SAnD.load_from_checkpoint(model_path)

In [ ]:
with initialize(version_base=None, config_path="../../configs"):
    cfg = compose(config_name="run_training.yaml", overrides=["data=cohort_a_r0.4"])

    print(cfg)

In [ ]:
wandb.init(mode="disabled")

In [ ]:
sepsis_pred_data = SepsisPredDataModule(cfg=cfg)


In [ ]:
sepsis_pred_data.setup(stage="fit")
sepsis_pred_data.setup(stage="test")

background_dl = sepsis_pred_data.train_dataloader()
inference_dl = sepsis_pred_data.test_dataloader()

In [ ]:
sepsis_pred_data.batch_size

In [ ]:
batch_background = next(iter(background_dl))

# batch = next(iter(inference_dl))
batch_features_ts, batch_features_stat, batch_labels = batch_background


In [ ]:

# background_samples = [batch_features_ts, batch_features_stat]

background_samples = [batch_features_ts.to("cuda"), batch_features_stat.to("cuda")]


In [ ]:
batch_explain = next(iter(inference_dl))
batch_explain_ts, batch_explain_stat, batch_explain_labels = batch_explain

explain_samples = [batch_explain_ts.to("cuda"), batch_explain_stat.to("cuda")]

In [ ]:
model.to("cuda")
model.eval()

with torch.no_grad():
    output = model(batch_features_ts.to("cuda"), batch_features_stat.to("cuda"))


In [ ]:
class ModelShapWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        # self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x_ts_in, x_static_in):
        # self.model.eval()

        if isinstance(self.model, GatedTransformer):
            x = self.model(x_ts_in, x_static_in, "test")
        else:
            x = self.model(x_ts_in, x_static_in)

        x = torch.nn.functional.sigmoid(x)

        return x



In [ ]:
model_shap = ModelShapWrapper(model=model)

In [ ]:
explainer = shap.DeepExplainer(
    # explainer = shap.Explainer(
    #         model=model,
    model=model_shap,
    data=background_samples,
    # masker=background_samples,
)

In [ ]:
explainer.feature_names = sepsis_pred_data.considered_features


In [ ]:
shap_values = explainer.shap_values(explain_samples, check_additivity=False)



In [ ]:
df_explanations_ts = pd.DataFrame(
    data=explain_samples[0].to("cpu").reshape(-1, len(sepsis_pred_data.considered_features)),
    columns=sepsis_pred_data.considered_features)


In [ ]:
df_explanations_ts.head()

In [ ]:
shap_explanation_ts = shap.Explanation(shap_values[0], base_values=explainer.expected_value,
                                       data=df_explanations_ts,
                                       feature_names=df_explanations_ts.columns)

In [ ]:
shap_explanation_ts.shape

In [ ]:
from tqdm.auto import tqdm

shap_values_batches = []
device = "cuda"
# with torch.no_grad():   already using eval mode
for batch in tqdm(inference_dl, desc="Calculating shap values"):
    sample_ts, sample_stat, sample_label = batch

    sample_ts = sample_ts.to(device)
    sample_stat = sample_stat.to(device)

    shap_values = explainer.shap_values([sample_ts, sample_stat], check_additivity=False)

    shap_values_batches.append(shap_values)

In [ ]:
shap_values_batches_flat_ts = [ndarray for sublist in shap_values_batches for ndarray in sublist[0]]
len(shap_values_batches_flat_ts)  # should be the number of total samples, e.g. 1529 for r0.4

In [ ]:
shap_values_batches_flat_ts[0].shape

In [ ]:
shap_values_batch_ts_mean = np.absolute(np.concatenate(shap_values_batches_flat_ts, axis=0).squeeze(2)).mean(axis=0)

In [ ]:
shap_values_batch_ts_mean.shape


In [ ]:
df_shap_values_ts_mean = pd.DataFrame(data=[shap_values_batch_ts_mean], columns=sepsis_pred_data.considered_features)

In [ ]:
fig_ts_mean, ax_ts_mean, _, _ = figure_setup(1, 1, width_cm=14, height_cm=10)
sns.barplot(df_shap_values_ts_mean, ax=ax_ts_mean)

ax_ts_mean.set_xticklabels(ax_ts_mean.get_xticklabels(), fontsize=4, rotation=90)


## SHAP Values Confusion Matrix

In [ ]:


results = []

inference_idx = 0

for batch in tqdm(inference_dl, desc="Determining"):
    sample_ts, sample_stat, sample_label = batch

    sample_ts = sample_ts.to(device)
    sample_stat = sample_stat.to(device)

    outputs = model_shap(sample_ts, sample_stat)

    threshold = 0.5  # threshold may vary
    preds = (outputs >= threshold).int()

    for i in range(len(sample_label)):
        results.append({
            "Inference Index": inference_idx,
            "Prediction": "sepsis" if preds[i].item() == 1 else "control",
            "Ground Truth": "sepsis" if sample_label[i].item() == 1 else "control"
        })
        inference_idx += 1



In [ ]:
df_test_inference_results = pd.DataFrame(results)
df_test_inference_results

In [ ]:
shap_values_batches_flat_ts


In [ ]:
shap_values_batches_flat_ts_stacked = np.stack(shap_values_batches_flat_ts)
shap_values_batches_flat_ts_stacked.shape


In [ ]:
shap_values_ts_all = np.mean(np.absolute(shap_values_batches_flat_ts_stacked), axis=1).squeeze(2)

In [ ]:
df_shap_ts_all = pd.DataFrame(data=shap_values_ts_all, columns=sepsis_pred_data.considered_features)
df_shap_ts_all.head()

In [ ]:
df_shap_ts_all_with_results = pd.concat([df_test_inference_results.drop(columns=["Inference Index"]), df_shap_ts_all],
                                        axis=1)
df_shap_ts_all_with_results.head()

### SHAP Importance Ground Truth = sepsis

In [ ]:
df_shap_ts_gt_sepsis = df_shap_ts_all_with_results.loc[df_shap_ts_all_with_results["Ground Truth"] == "sepsis", :].drop(
    columns=["Ground Truth", "Prediction"])
df_shap_ts_gt_sepsis.shape

In [ ]:
fig_ts_mean, ax_ts_mean, _, _ = figure_setup(1, 1, width_cm=14, height_cm=10)
sns.barplot(df_shap_ts_gt_sepsis, ax=ax_ts_mean, orient="h", capsize=0.05, err_kws={"linewidth": 0.7})

ax_ts_mean.set_yticklabels(ax_ts_mean.get_yticklabels(), fontsize=4,
                           #rotation=90
                           )

In [ ]:
df_shap_ts_gt_control = df_shap_ts_all_with_results.loc[df_shap_ts_all_with_results["Ground Truth"] == "control",
                        :].drop(columns=["Ground Truth", "Prediction"])
df_shap_ts_gt_control.shape


In [ ]:
fig_ts_mean, ax_ts_mean, _, _ = figure_setup(1, 1, width_cm=14, height_cm=10)
sns.barplot(df_shap_ts_gt_control, ax=ax_ts_mean, orient="h", capsize=0.05, err_kws={"linewidth": 0.7})

ax_ts_mean.set_yticklabels(ax_ts_mean.get_yticklabels(), fontsize=4,
                           #rotation=90
                           )

In [ ]:
df_shap_ts_pred_sepsis = df_shap_ts_all_with_results.loc[df_shap_ts_all_with_results["Prediction"] == "sepsis", :].drop(
    columns=["Ground Truth", "Prediction"])
df_shap_ts_pred_sepsis.shape


In [ ]:
fig_ts_mean, ax_ts_mean, _, _ = figure_setup(1, 1, width_cm=14, height_cm=10)
sns.barplot(df_shap_ts_pred_sepsis, ax=ax_ts_mean, orient="h", capsize=0.05, err_kws={"linewidth": 0.7})

ax_ts_mean.set_yticklabels(ax_ts_mean.get_yticklabels(), fontsize=4,
                           #rotation=90
                           )


In [ ]:

df_shap_ts_pred_control = df_shap_ts_all_with_results.loc[df_shap_ts_all_with_results["Prediction"] == "control",
                          :].drop(columns=["Ground Truth", "Prediction"])
df_shap_ts_pred_control.shape


In [ ]:
mean, ax_ts_mean, _, _ = figure_setup(1, 1, width_cm=14, height_cm=10)
sns.barplot(df_shap_ts_pred_control, ax=ax_ts_mean, orient="h", capsize=0.05, err_kws={"linewidth": 0.7})

ax_ts_mean.set_yticklabels(ax_ts_mean.get_yticklabels(), fontsize=4,
                           #rotation=90
                           )


## Cohort SHAP Feature Importance

In [ ]:
df_test = sepsis_pred_data.df_test

In [ ]:
df_test_unique = df_test.drop_duplicates(subset=[sepsis_pred_data.sample_index_column, "patientunitstayid"])
df_test_unique.head()

In [ ]:
df_test_sample_indices_id = df_test_unique.loc[:, [sepsis_pred_data.sample_index_column, "patientunitstayid"]]
df_test_sample_indices_id.head()

In [ ]:
df_static = sepsis_pred_data.df_static
df_static.head()

In [ ]:
df_static_test = df_static.loc[df_static["patientunitstayid"].isin(df_test_sample_indices_id["patientunitstayid"])]
df_static_test["unittype"].describe()

In [ ]:
df_static_test

In [ ]:
df_static_ts_test = pd.merge(df_test_sample_indices_id, df_static_test, on="patientunitstayid", how="left")

In [ ]:
df_test_cohort_sample_idx = df_static_ts_test.loc[:,[sepsis_pred_data.sample_index_column, "unittype"]]
df_test_cohort_sample_idx["unittype"].value_counts()

In [ ]:
df_static["unittype"].value_counts()

In [ ]:
df = df_test_cohort_sample_idx[df_test_cohort_sample_idx["unittype"] != ""]
# df2 = df_test_cohort_sample_idx[df_test_cohort_sample_idx["unittype"] == crit1eria_2]

